# B4 — Fine-tuning LoRA d'un petit LLM pour rédiger les rapports d'audit

**Objectif (cahier des charges, B4).** Adapter un petit modèle open source (Qwen 0,5B–1,5B Instruct) à la rédaction de rapports d'audit, puis comparer sur **les mêmes alertes de test** :

| Configuration | Ce qu'elle mesure |
|---|---|
| Prompt seul | le modèle de base avec un prompt soigné (structure + règles), sans entraînement |
| Prompt seul + RAG | idem, avec des extraits réglementaires (isole l'effet du RAG) |
| LoRA (rang 4, 8, 16) | l'apport du fine-tuning, et l'effet du rang |
| LoRA + RAG | le LoRA avec des extraits réglementaires ajoutés au prompt |

**Mesures automatiques** (`audit/report_checks.py` et `audit/lora_utils.py`) : exactitude des faits (valeurs de l'alerte reprises, rien d'inventé), structure en 5 sections, citation des textes (dont la Recommandation 20 du GAFI). **La clarté et la justesse des citations se jugent à la main** sur l'échantillon généré à la fin.

**Avant de lancer**
1. Exécution → Modifier le type d'exécution → **GPU T4**.
2. Les fichiers de `audit/` et `audit/instruction_data/` doivent être **poussés** sur la branche indiquée plus bas.
3. Première exécution avec `QUICK = True` (quelques minutes) pour vérifier que tout tourne, puis `QUICK = False` pour le vrai run.

In [ ]:
# ===== PARAMÈTRES (à adapter) =====
REPO_URL   = "https://github.com/hafid700/FinTrust-AI-Project.git"  # dépôt privé : https://<TOKEN>@github.com/hafid700/FinTrust-AI-Project.git
BRANCH     = "feature-B"                       # nom exact de ta branche
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"      # ou "Qwen/Qwen2.5-1.5B-Instruct"
RANKS      = [4, 8, 16]                        # rangs LoRA à comparer
EPOCHS     = 2
LR         = 2e-4
BATCH_SIZE = 4      # micro-lot ; passe à 2 en cas de "CUDA out of memory"
GRAD_ACCUM = 4      # lot effectif = BATCH_SIZE x GRAD_ACCUM = 16
EVAL_BS    = 8      # lot pour la génération
RAG_K      = 2      # nombre d'extraits réglementaires ajoutés au prompt
SEED       = 42
QUICK      = True   # True = test rapide (peu d'exemples) ; False = vrai run

## 1. Installation, GPU et dépôt
Les versions installées sont affichées : **recopie-les dans `requirements.txt`** après le premier run réussi (le cahier demande des versions figées).

In [ ]:
!pip -q install -U transformers peft accelerate sentence-transformers faiss-cpu pypdf pandas matplotlib

In [ ]:
import importlib.metadata as md_
import torch

for pkg in ["torch", "transformers", "peft", "accelerate", "sentence-transformers", "faiss-cpu", "pypdf"]:
    print(f"{pkg}=={md_.version(pkg)}")
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > T4 GPU"
print("GPU :", torch.cuda.get_device_name(0))

In [ ]:
import os
if not os.path.isdir("FinTrust-AI-Project"):
    !git clone -b {BRANCH} {REPO_URL}
%cd FinTrust-AI-Project
!ls audit audit/instruction_data

## 2. Corpus réglementaire et index FAISS (B1)
Le RAG consulte ces documents publics ; ils ne servent pas à l'entraînement. Si un téléchargement échoue, dépose le PDF à la main dans `audit/corpus/` et relance la cellule.

In [ ]:
import os, requests

os.makedirs("audit/corpus", exist_ok=True)
CORPUS = {
    "FATF_Recommendations_2012.pdf": "https://www.fatf-gafi.org/content/dam/fatf-gafi/recommendations/FATF%20Recommendations%202012.pdf.coredownload.inline.pdf",
    "RGPD.pdf": "https://wallex.wallonie.be/files/medias/10/RGPD.pdf",
    "Guide-LBC-FT-FR-03_25.pdf": "https://cnasnu.justice.gov.ma/wp-content/uploads/2025/03/Guide-LBC-FT-FR-03_25.pdf",
    "Lutte-contre-le-blanchiment-de-capitaux.pdf": "https://adala.justice.gov.ma/api/uploads/2024/02/29/Lutte%20contre%20le%20blanchiment%20de%20capitaux-1709213287982.pdf",
}
for name, url in CORPUS.items():
    path = f"audit/corpus/{name}"
    if os.path.exists(path) and os.path.getsize(path) > 10_000:
        print("déjà présent :", name)
        continue
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=120)
        r.raise_for_status()
        open(path, "wb").write(r.content)
        print("OK     :", name, len(r.content) // 1024, "Ko")
    except Exception as e:
        print("ÉCHEC  :", name, "->", e, "| dépose le PDF à la main dans audit/corpus/")

if not os.path.exists("audit/rag_index/index.faiss"):
    !python audit/build_rag_index.py
else:
    print("index FAISS déjà construit")

## 3. Données, tokenizer et mesure des longueurs
`encode_example` masque le prompt : **la perte ne porte que sur le rapport**. On mesure les longueurs pour fixer `max_new_tokens` (génération) et vérifier que les séquences tiennent en mémoire.

In [ ]:
import os, sys, json, time, gc
sys.path.insert(0, "audit")
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import lora_utils as L
from search_rag import search

L.set_seed(SEED)
train_recs = L.load_jsonl("audit/instruction_data/train.jsonl")
val_recs   = L.load_jsonl("audit/instruction_data/val.jsonl")
test_recs  = L.load_jsonl("audit/instruction_data/test.jsonl")

CFG = dict(ranks=list(RANKS), epochs=EPOCHS)
if QUICK:
    train_recs, val_recs, test_recs = train_recs[:16], val_recs[:8], test_recs[:8]
    CFG = dict(ranks=list(RANKS)[:2], epochs=1)
    print("*** MODE TEST RAPIDE : résultats NON exploitables pour le rapport ***")
print(f"train={len(train_recs)} val={len(val_recs)} test={len(test_recs)} | rangs={CFG['ranks']} | époques={CFG['epochs']}")

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
train_ds = L.build_dataset(tok, train_recs)
val_ds   = L.build_dataset(tok, val_recs)
stats = L.token_stats(train_ds)
print({k: round(v) for k, v in stats.items()})
MAX_NEW_TOKENS = int(stats["report_tokens_max"] * 1.2)
print("max_new_tokens =", MAX_NEW_TOKENS)

## 4. Contexte RAG des alertes de test
L'alerte ne contient pas de texte : la question posée à FAISS dépend du **schéma détecté** (cycle, fan-in, fan-out, none) et de la déclaration de soupçon. **Toutes les alertes d'un même schéma reçoivent donc le même contexte** (limite à mentionner dans l'analyse).

In [ ]:
test_alerts = [json.loads(r["input"]) for r in test_recs]
contexts = L.build_contexts(test_alerts, search, k=RAG_K)
print(contexts[test_alerts[0]["tx_id"]][:1200])

## 5. Fonctions d'aide et configurations de référence (sans entraînement)
Le « prompt seul » reçoit la structure du rapport et les règles : on mesure ce que le LoRA apporte **en plus** d'un prompt soigné, pas contre un prompt naïf.

In [ ]:
def load_base():
    """Recharge un modèle de base frais (float16, GPU)."""
    try:
        m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16)
    except TypeError:  # anciennes versions de transformers
        m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
    return m.to("cuda")

RESULTS, ROWS, LOSS_CURVES = {}, [], {}

def record(name, rank, res, train=None):
    row = {
        "configuration": name,
        "rang": rank,
        "params_entrainables": train["trainable_params"] if train else 0,
        "temps_entrainement_s": round(train["train_seconds"]) if train else 0,
        "pic_memoire_gpu_Go": round(train["peak_gpu_gb"], 1) if train and train["peak_gpu_gb"] is not None else None,
        "perte_validation": round(train["val_losses"][-1], 4) if train and train["val_losses"] else None,
        "temps_generation_s": round(res["gen_seconds"]),
        **res["summary"],
    }
    ROWS.append(row)
    RESULTS[name] = res
    print(name, "->", res["summary"])

EVAL = dict(max_new_tokens=MAX_NEW_TOKENS, batch_size=EVAL_BS)

In [ ]:
model = load_base()
res = L.evaluate_model(model, tok, test_recs, instruction=L.PROMPT_ONLY_INSTRUCTION, **EVAL)
record("Prompt seul", None, res)
res = L.evaluate_model(model, tok, test_recs, instruction=L.PROMPT_ONLY_INSTRUCTION, contexts=contexts, **EVAL)
record("Prompt seul + RAG", None, res)
del model; gc.collect(); torch.cuda.empty_cache()

## 6. Fine-tuning LoRA, rang par rang
Pour chaque rang : modèle de base frais → entraînement → sauvegarde de l'adaptateur → évaluation sans RAG puis avec RAG.
`alpha = 2 × rang` garde le facteur d'échelle `alpha/rang` constant : la comparaison isole l'effet de la **capacité** (le rang).

In [ ]:
os.makedirs("audit/lora_adapters", exist_ok=True)
for rank in CFG["ranks"]:
    print(f"\n===== LoRA rang {rank} =====")
    base = load_base()
    tr = L.train_lora(base, tok, train_ds, val_ds, rank, epochs=CFG["epochs"], lr=LR,
                      batch_size=BATCH_SIZE, grad_accum=GRAD_ACCUM, seed=SEED)
    model = tr["model"]
    model.save_pretrained(f"audit/lora_adapters/rank_{rank}")
    LOSS_CURVES[rank] = tr["history"]

    res = L.evaluate_model(model, tok, test_recs, **EVAL)
    record(f"LoRA r={rank}", rank, res, tr)
    res = L.evaluate_model(model, tok, test_recs, contexts=contexts, **EVAL)
    record(f"LoRA r={rank} + RAG", rank, res, tr)

    del model, base, tr
    gc.collect(); torch.cuda.empty_cache()

## 7. Tableau de comparaison
Colonnes : `facts_ok_%` (tous les faits repris, rien d'inventé), `mean_problems` (écarts par rapport, 0 = parfait), `sections_ok_%` (structure), `cite_any_%` / `cite_R20_%` (citations). Avec **50 exemples de test**, un écart de 2 points correspond à un seul rapport : ne le sur-interprète pas.

In [ ]:
df = pd.DataFrame(ROWS)
suffix = "_quick" if QUICK else ""
os.makedirs("audit/results", exist_ok=True)
df.to_csv(f"audit/results/b4_comparison{suffix}.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
df

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].barh(df["configuration"][::-1], df["facts_ok_%"][::-1])
ax[0].set_xlabel("Rapports sans aucun écart factuel (%)"); ax[0].set_xlim(0, 100)
ax[0].set_title("Exactitude des faits")
for rank, hist in LOSS_CURVES.items():
    ax[1].plot([s for s, _ in hist], [l for _, l in hist], label=f"r={rank}")
ax[1].set_xlabel("Étape d'optimisation"); ax[1].set_ylabel("Perte (entraînement)")
ax[1].set_title("Courbes d'entraînement"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"audit/results/b4_comparison{suffix}.png", dpi=150); plt.show()

## 8. Échantillon pour la relecture manuelle
Le cahier exige une **relecture manuelle** : ouvre `audit/results/b4_samples.md` et note chaque exemple (clarté, justesse des citations). Les mesures automatiques ne voient ni un texte confus ni une citation hors sujet.

In [ ]:
lora_only = df[df["configuration"].str.startswith("LoRA r=") & ~df["configuration"].str.contains("RAG")]
best = lora_only.sort_values(["facts_ok_%", "perte_validation"], ascending=[False, True]).iloc[0]
best_rank = int(best["rang"])
show = {k: RESULTS[k] for k in ["Prompt seul", f"LoRA r={best_rank}", f"LoRA r={best_rank} + RAG"]}
L.write_samples_markdown(f"audit/results/b4_samples{suffix}.md", show, test_recs, n=3)
print("meilleur rang (faits puis perte validation) :", best_rank)
print(open(f"audit/results/b4_samples{suffix}.md", encoding="utf-8").read()[:3500])

## 9. Sauvegarde des résultats et de l'adaptateur
Les CSV / PNG / Markdown sont légers : commit-les dans `audit/results/`. Un adaptateur LoRA pèse quelques dizaines de Mo : garde-le sur Drive ou en release plutôt que dans l'historique Git si le dépôt est partagé.

In [ ]:
!cd audit && zip -qr b4_outputs.zip lora_adapters results
from google.colab import files
files.download("audit/b4_outputs.zip")

## Limites à écrire dans l'analyse finale
- **Dataset synthétique** : les rapports de référence sortent de modèles de texte. Le LoRA apprend surtout un format et un style ; la comparaison mesure donc aussi la fidélité au format, pas la qualité rédactionnelle générale.
- **Contexte RAG par schéma** : 4 contextes distincts pour tout le jeu de test (le LoRA n'a pas été entraîné avec du contexte).
- **Peu d'exemples de test, une seule graine** : les petits écarts ne sont pas significatifs.
- **Mesures automatiques = proxys** : seule la relecture manuelle juge clarté et justesse des citations.
- **Alertes factices** : à refaire avec les vraies sorties de A (B2 → B3 → B4) à l'intégration.